In [ ]:

# Kaggle setup (run this first)
!pip install -q transformers datasets peft accelerate bitsandbytes


In [ ]:

import random
import torch
from datasets import Dataset, DatasetDict


In [ ]:

from huggingface_hub import login

HF_TOKEN = "your_huggingface_token_here"
login(HF_TOKEN)


In [ ]:

# Replace this with your Stage 9 dataset
data = [{"text": f"INPUT: log {i}\nOUTPUT: analysis {i}"} for i in range(500)]

random.shuffle(data)
split = int(0.9 * len(data))

train_data = data[:split]
val_data = data[split:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(train_dataset, val_dataset)


In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)


In [ ]:

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

train_tokenized = train_dataset.map(tokenize, batched=True, remove_columns=["text"])
val_tokenized = val_dataset.map(tokenize, batched=True, remove_columns=["text"])

train_tokenized.set_format("torch")
val_tokenized.set_format("torch")


In [ ]:

from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="/kaggle/working/outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()


In [ ]:

model.save_pretrained("/kaggle/working/adapter")
tokenizer.save_pretrained("/kaggle/working/adapter")

print("Saved to /kaggle/working/adapter")
